# 0) Setup: mounts, config, helpers

In [ ]:
# ==== STEP 0: INSTALL DEPENDENCIES (COLAB / PYTHON 3.12 SAFE) ====

# Core scientific stack + genomics
!pip install -q biopython==1.83 pandas==2.2.2 pyarrow==17.0.0 \
                fastparquet==2024.5.0 intervaltree==3.1.0 \
                datasets==2.21.0 transformers==4.44.2 sentencepiece==0.2.0 \
                beautifulsoup4==4.12.3 datasketch==1.6.5 requests==2.32.4

# Compatibility layer for Colab (ensure gcsfs/fsspec align with 2025 releases)
!pip install -q fsspec==2025.3.0 gcsfs==2025.3.0 polars==1.28.2

# For progress tracking and optional utilities
!pip install -q tqdm==4.66.4 rich==13.9.2

# System packages (for tRNAscan-SE, zlib, SSL headers)
!apt-get -y update -qq && apt-get -y install -qq \
        trnascan-se libcurl4-openssl-dev libssl-dev zlib1g-dev libbz2-dev

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

In [ ]:
# ==== 0) SETUP: DRIVE + CONFIG + HELPERS (idempotent) ====

from google.colab import drive
drive.mount('/content/drive')

import os, io, gzip, csv, json, time, shutil, math, random, re, hashlib
from pathlib import Path
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from dataclasses import dataclass
from datetime import datetime
from Bio import SeqIO, Entrez

# ---- Project roots on Drive
ROOT     = Path('/content/drive/MyDrive/MitoGPT/curated/v3.0')
RAW      = ROOT/'raw';       RAW.mkdir(parents=True, exist_ok=True)
CACHE    = ROOT/'cache';     CACHE.mkdir(parents=True, exist_ok=True)
DATASET  = ROOT/'dataset';   DATASET.mkdir(parents=True, exist_ok=True)
MANIFEST = ROOT/'manifests'; MANIFEST.mkdir(parents=True, exist_ok=True)
REPORTS  = ROOT/'reports';   REPORTS.mkdir(parents=True, exist_ok=True)
TOKDIR   = ROOT/'tokenizer/dnabert2_sentencepiece'; TOKDIR.mkdir(parents=True, exist_ok=True)
(RAW/'mitomap').mkdir(exist_ok=True, parents=True)
(RAW/'references').mkdir(exist_ok=True, parents=True)
(RAW/'ncbi_organelle').mkdir(exist_ok=True, parents=True)
(RAW/'mitobank').mkdir(exist_ok=True, parents=True)

# ---- Config (ensure email is set!)
CONFIG = {
    "email_for_ncbi": "tat-ching.kong@cnrs.fr",    # <= update if needed
    "fetch_cross_species": True,
    "fetch_human_mt_sequences": True,
    "download_conservation_tracks": False,         # we use UCSC JSON API instead of bigWig
    "synthetic_mut_rate": 2e-3,
    "synthetic_n": 100_000,                        # scale here (start lower to test)
    "near_dup_threshold": 0.99,                    # very strict near-dup => we patch bands below
}
Entrez.email = CONFIG["email_for_ncbi"]
TODAY = datetime.utcnow().date().isoformat()

# ---- Simple cached downloader (to CACHE/)
def cached_get(url:str, timeout=120, max_tries=3, as_bytes=True):
    h = hashlib.sha256(url.encode()).hexdigest()
    ext = '.bin'
    if url.endswith('.gz'):   ext = '.gz'
    if url.endswith('.gff'):  ext = '.gff'
    if url.endswith('.vcf'):  ext = '.vcf'
    if url.endswith('.fna') or url.endswith('.fa') or url.endswith('.fasta'): ext = '.fasta'
    f = CACHE/(h+ext)
    if f.exists() and f.stat().st_size > 128:  # sanity
        return str(f)
    for _ in range(max_tries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp = str(f)+'.tmp'
                with open(tmp, 'wb') as o:
                    for ch in r.iter_content(chunk_size=1<<20):
                        if ch: o.write(ch)
                os.replace(tmp, f)
                break
        except Exception:
            time.sleep(2)
    return str(f)

# ---- Hive writer
def write_partitioned(rows, root:Path):
    if not rows:
        return
    tbl = pa.Table.from_pylist(rows)
    pq.write_to_dataset(
        tbl,
        root_path=str(root),
        partition_cols=['task','tier','source','species','date'],
        use_dictionary=True,
        compression='zstd'
    )

# ---- MLAA JSON col wrapper
def to_json(obj)->str:
    return json.dumps(obj, separators=(',',':'))

# ---- GFF utils
def read_gff_gz(path:Path) -> pd.DataFrame:
    # handles plain .gff or .gz
    if str(path).endswith('.gz'):
        with gzip.open(path, 'rt', errors='replace') as f:
            lines = [l for l in f if l.strip() and not l.startswith('#')]
    else:
        with open(path, 'rt', errors='replace') as f:
            lines = [l for l in f if l.strip() and not l.startswith('#')]
    recs=[]
    for ln in lines:
        arr = ln.rstrip('\n').split('\t')
        if len(arr) < 9:
            continue
        recs.append({
            "seqid": arr[0], "source": arr[1], "type": arr[2],
            "start": int(arr[3]), "end": int(arr[4]),
            "score": arr[5], "strand": arr[6], "phase": arr[7],
            "attributes": arr[8]
        })
    return pd.DataFrame(recs)

def parse_attributes(s:str):
    d={}
    for part in s.split(';'):
        if '=' in part:
            k,v = part.split('=',1)
            d[k]=v
    return d

# ---- Lightweight duplication filter (exact + very near-dup) [PATCHED]
from datasketch import MinHash, MinHashLSH

class DuplicateFilter:
    """
    Near-dup filter that works even with very high thresholds.
    If threshold >= 0.97 we bypass the 'threshold' constructor and
    explicitly set (b, r) so that b >= 2 (required by datasketch).
    """
    def __init__(self, threshold=0.95, num_perm=128, k=7):
        self.k = k
        self.num_perm = num_perm
        self._seen = set()
        self._i = 0

        if threshold >= 0.97:
            # Choose r=4 rows per band; compute bands b so that b >= 2
            r = 4
            b = max(2, num_perm // r)
            self.lsh = MinHashLSH(params=(b, r), num_perm=num_perm)
        else:
            self.lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    def _kshingles(self, s: str):
        s = s.upper()
        step = self.k  # non-overlapping k-shingles for speed
        for i in range(0, max(0, len(s) - self.k + 1), step):
            yield s[i:i+self.k]

    def exact_seen(self, s: str) -> bool:
        h = hashlib.md5(s.encode()).hexdigest()
        if h in self._seen:
            return True
        self._seen.add(h)
        return False

    def near_dup(self, s: str) -> bool:
        m = MinHash(num_perm=self.num_perm)
        for sh in self._kshingles(s):
            m.update(sh.encode())
        res = self.lsh.query(m)
        self.lsh.insert(f'item-{self._i}', m)
        self._i += 1
        return len(res) > 0

# Instantiate with safe params (handles 0.99 cleanly)
dupfilter = DuplicateFilter(
    threshold=min(CONFIG.get("near_dup_threshold", 0.95), 0.99),
    num_perm=128,
    k=7
)

print("✅ Setup ready. Roots:", ROOT)

Mounted at /content/drive


/tmp/ipython-input-2340975267.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TODAY = datetime.utcnow().date().isoformat()


✅ Setup ready. Roots: /content/drive/MyDrive/MitoGPT/curated/v3.0


# 1) Tokenizer + robust rCRS fetch (fallbacks)

In [ ]:
# ==== 1) TOKENIZER + rCRS (robust) ====
from transformers import AutoTokenizer

RCRS_FASTA_URL = "https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/110/GCF_000001405.38-GRCh38.p12/mitochondrion/NC_012920.1.fna"
RCRS_FASTA_DST = RAW/'references'/'rCRS.fasta'

def _looks_like_fasta(p: Path) -> bool:
    try:
        if not p.exists() or p.stat().st_size < 200:  # mtDNA is 16.5kb; tiny file is suspicious
            return False
        with open(p, 'rt', errors='ignore') as f:
            head = f.read(1024)
        return head.lstrip().startswith('>')
    except Exception:
        return False

def _download_to(path: Path, url: str, timeout=120) -> bool:
    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            tmp = str(path) + ".tmp"
            with open(tmp, "wb") as o:
                for ch in r.iter_content(chunk_size=1<<20):
                    if ch: o.write(ch)
            os.replace(tmp, path)
        return True
    except Exception as e:
        print(f"[rCRS] direct download failed: {e}")
        return False

def _fetch_rCRS_fasta() -> Path:
    """Try cache of FTP → direct download → Entrez efetch (fallback). Returns path to a valid FASTA."""
    # 1) Try cached FTP
    try:
        f_cached = Path(cached_get(RCRS_FASTA_URL))
    except Exception as e:
        print("[rCRS] cached_get error:", e)
        f_cached = Path("/nonexistent")

    if f_cached.exists() and _looks_like_fasta(f_cached):
        shutil.copyfile(f_cached, RCRS_FASTA_DST)
        return RCRS_FASTA_DST

    # 2) Try direct download now (not via cache)
    if _download_to(RCRS_FASTA_DST, RCRS_FASTA_URL) and _looks_like_fasta(RCRS_FASTA_DST):
        return RCRS_FASTA_DST

    # 3) Fallback: Entrez efetch (reliable)
    try:
        print("[rCRS] Falling back to Entrez efetch ...")
        h = Entrez.efetch(db="nuccore", id="NC_012920.1", rettype="fasta", retmode="text")
        fasta = h.read(); h.close()
        RCRS_FASTA_DST.write_text(fasta)
        if _looks_like_fasta(RCRS_FASTA_DST):
            return RCRS_FASTA_DST
    except Exception as e:
        print("[rCRS] Entrez efetch failed:", e)

    # 4) Last resort: try GenBank then write FASTA from SeqIO
    try:
        h = Entrez.efetch(db="nuccore", id="NC_012920.1", rettype="gb", retmode="text")
        gb_txt = h.read(); h.close()
        (RAW/'references'/'rCRS.gb').write_text(gb_txt)
        rec = SeqIO.read(str(RAW/'references'/'rCRS.gb'), 'genbank')
        SeqIO.write(rec, str(RCRS_FASTA_DST), 'fasta')
        if _looks_like_fasta(RCRS_FASTA_DST):
            return RCRS_FASTA_DST
    except Exception as e:
        print("[rCRS] GenBank fallback failed:", e)

    raise RuntimeError("Unable to retrieve rCRS FASTA from all sources.")

# Fetch & parse rCRS
fa_path = _fetch_rCRS_fasta()
try:
    rCRS_seq = str(next(SeqIO.parse(str(fa_path), 'fasta')).seq).upper()
except StopIteration:
    # Very defensive: re-download via Entrez FASTA
    print("[rCRS] FASTA parse empty; retrying via Entrez ...")
    h = Entrez.efetch(db="nuccore", id="NC_012920.1", rettype="fasta", retmode="text")
    fasta = h.read(); h.close()
    RCRS_FASTA_DST.write_text(fasta)
    rCRS_seq = str(next(SeqIO.parse(str(RCRS_FASTA_DST), 'fasta')).seq).upper()

# DNABERT-2 tokenizer (pin for reproducibility)
tok = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M")
tok.save_pretrained(str(TOKDIR))

print("rCRS path:", fa_path)
print("rCRS length:", len(rCRS_seq))
print("Tokenizer saved to:", TOKDIR)


[rCRS] direct download failed: 404 Client Error: Not Found for url: https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/110/GCF_000001405.38-GRCh38.p12/mitochondrion/NC_012920.1.fna
[rCRS] Falling back to Entrez efetch ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


rCRS path: /content/drive/MyDrive/MitoGPT/curated/v3.0/raw/references/rCRS.fasta
rCRS length: 16569
Tokenizer saved to: /content/drive/MyDrive/MitoGPT/curated/v3.0/tokenizer/dnabert2_sentencepiece


# 2) Download sources

In [ ]:
# ==== 2) DOWNLOAD SOURCES (Drive-safe copies, no os.link) ====
MITOMAP = {
  "polymorphisms_gff": "https://mitomap.org/cgi-bin/polymorphisms.cgi?format=gff",
  "polymorphisms_vcf": "https://mitomap.org/cgi-bin/polymorphisms.cgi?format=vcf",
  "disease_gff":       "https://mitomap.org/cgi-bin/disease.cgi?format=gff"
}
for name, url in MITOMAP.items():
    path = RAW/'mitomap'/f"{name}.gz"
    if not path.exists():
        src = cached_get(url)
        shutil.copyfile(src, path)

print("MITOMAP saved in:", RAW/'mitomap')

# Broaden cross-species harvest for more data
if CONFIG["fetch_cross_species"]:
    q = '(mitochondrion[Filter]) AND ("complete genome"[Title] OR complete[Title]) AND 2000:30000[SLEN]'
    h = Entrez.esearch(db="nuccore", term=q, retmax=20000)
    ids = Entrez.read(h)['IdList']; h.close()
    print("Cross-species IDs:", len(ids))
    for i in range(0, len(ids), 250):
        ef = Entrez.efetch(db="nuccore", id=ids[i:i+250], rettype="gb", retmode="text")
        (RAW/'ncbi_organelle'/f'organelle_{i//250:04d}.gb').write_text(ef.read())
        ef.close()

if CONFIG["fetch_human_mt_sequences"]:
    h = Entrez.esearch(db="nuccore", term='("Homo sapiens"[Organism]) AND mitochondrion[Filter] AND ("complete genome"[Title] OR complete[Title])', retmax=25000)
    hids = Entrez.read(h)['IdList']; h.close()
    print("Human mtDNA IDs:", len(hids))
    for i in range(0, len(hids), 500):
        ef = Entrez.efetch(db="nuccore", id=hids[i:i+500], rettype="fasta", retmode="text")
        (RAW/'mitobank'/f'human_mt_{i//500:04d}.fasta').write_text(ef.read()); ef.close()


MITOMAP saved in: /content/drive/MyDrive/MitoGPT/curated/v3.0/raw/mitomap
Cross-species IDs: 20000


KeyboardInterrupt: 

# 3) MITOMAP → tidy tables

In [ ]:
# ==== 3) LOAD & TIDY MITOMAP (auto-detect gzip; optional normalize) ====
import gzip

def _is_gzip(path: Path) -> bool:
    try:
        with open(path, 'rb') as f:
            return f.read(2) == b'\x1f\x8b'
    except Exception:
        return False

def _open_text_auto(path: Path):
    """Open text file whether it's gzipped or not."""
    if str(path).endswith('.gz') and _is_gzip(path):
        return gzip.open(path, 'rt', errors='replace')
    return open(path, 'rt', errors='replace')

def read_gff_auto(path: Path) -> pd.DataFrame:
    recs = []
    with _open_text_auto(path) as f:
        for ln in f:
            if not ln or ln.startswith('#') or not ln.strip():
                continue
            arr = ln.rstrip('\n').split('\t')
            if len(arr) < 9:
                continue
            recs.append({
                "seqid": arr[0], "source": arr[1], "type": arr[2],
                "start": int(arr[3]), "end": int(arr[4]),
                "score": arr[5], "strand": arr[6], "phase": arr[7],
                "attributes": arr[8]
            })
    return pd.DataFrame(recs, columns=['seqid','source','type','start','end','score','strand','phase','attributes'])

def parse_attributes(s: str):
    d = {}
    for part in s.split(';'):
        if '=' in part:
            k, v = part.split('=', 1)
            d[k] = v
    return d

def explode_attrs(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.assign(start=pd.Series(dtype='int64'), end=pd.Series(dtype='int64'))
    meta = df['attributes'].map(parse_attributes).apply(pd.Series)
    out  = pd.concat([df.drop(columns=['attributes']), meta], axis=1)
    out['start'] = pd.to_numeric(out['start'], errors='coerce').astype('Int64')
    out['end']   = pd.to_numeric(out['end'],   errors='coerce').astype('Int64')
    return out

# --- OPTIONAL: normalize on-disk files so .gz are truly gzipped (one-time) ---
def normalize_gz_inplace(path: Path):
    """If file endswith .gz but is not gzipped, re-write a gzipped version in-place."""
    if str(path).endswith('.gz') and not _is_gzip(path):
        tmp = str(path) + '.tmpgz'
        with open(path, 'rb') as src, gzip.open(tmp, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        os.replace(tmp, path)

for p in (RAW/'mitomap').glob('*.gz'):
    normalize_gz_inplace(p)

# --- Load your already-downloaded files (works whether gzipped or not) ---
poly_path = RAW/'mitomap'/'polymorphisms_gff.gz'
dis_path  = RAW/'mitomap'/'disease_gff.gz'

poly_gff = read_gff_auto(poly_path)
dis_gff  = read_gff_auto(dis_path)

poly = explode_attrs(poly_gff)
dis  = explode_attrs(dis_gff)

print("MITOMAP polymorphisms:", len(poly), "disease rows:", len(dis))
print("First polymorphism row:", dict(poly.iloc[0]) if len(poly) else "NA")
print("First disease row:", dict(dis.iloc[0]) if len(dis) else "NA")

# 4) Control-region loci & replication origins (explicit) + UCSC JSON conservation

In [ ]:
# ==== 4) CONTROL REGION & CONSERVATION (no bigWig; UCSC JSON API) ====

# Wrap-around aware coords (rCRS length 16569)
# D-loop spans 16024..16569 and 1..576
CR_LOCI = [
    {"name": "D-loop_A", "start": 16024-1, "end": 16569},  # 0-based inclusive-exclusive in our code later
    {"name": "D-loop_B", "start": 0,        "end": 576},
    {"name": "HVR1",     "start": 16024-1, "end": 16365},
    {"name": "HVR2",     "start": 73-1,    "end": 340},
    {"name": "CSB1",     "start": 213-1,   "end": 235},
    {"name": "CSB2",     "start": 299-1,   "end": 315},
    {"name": "CSB3",     "start": 346-1,   "end": 363},
    {"name": "LSP",      "start": 407-1,   "end": 440},
    {"name": "HSP1",     "start": 561-1,   "end": 585},
    {"name": "TAS",      "start": 16157-1, "end": 16172}
]
REPL_ORIGINS = [
    {"name":"OriH","type":"replication_origin","start":110-1,"end":441},    # includes promoters
    {"name":"OriL","type":"replication_origin","start":5721-1,"end":5798}   # WANCY region vicinity
]

# UCSC JSON API (hg38 chrM or MT) — avoids pybigwig
UCSC_JSON = "https://api.genome.ucsc.edu/list/chromosomes/hg38"
UCSC_SIGNAL_BASE = "https://api.genome.ucsc.edu/getData/track"
UCSC_PC = "phastCons20way"
UCSC_PP = "phyloP20way"

def _ucsc_chrom_mt():
    try:
        r = requests.get(UCSC_JSON, timeout=30).json()
        chroms = r.get('chromosomes', [])
        if 'chrM' in chroms: return 'chrM'
        if 'MT'   in chroms: return 'MT'
    except Exception:
        pass
    return 'chrM'

_UCSC_MT = _ucsc_chrom_mt()
print("Conservation provider: UCSC JSON API; chrom:", _UCSC_MT)

def conservation_scores_chrM(start:int, end:int, thinning=300):
    # UCSC getData/track?genome=hg38;track=phastCons20way;chrom=chrM;start=..;end=..
    try:
        pc = requests.get(f"{UCSC_SIGNAL_BASE}", params={
            "genome":"hg38", "track":UCSC_PC, "chrom":_UCSC_MT, "start":start, "end":end
        }, timeout=30).json()
        pp = requests.get(f"{UCSC_SIGNAL_BASE}", params={
            "genome":"hg38", "track":UCSC_PP, "chrom":_UCSC_MT, "start":start, "end":end
        }, timeout=30).json()
        pc_vals = pc.get(UCSC_PC, [])
        pp_vals = pp.get(UCSC_PP, [])
        # map into arrays indexed by position
        # UCSC returns list of {chrom,start,end,score} segments
        L=end-start
        out=[]
        for i in range(L):
            pos = start+i
            # naive nearest segment lookup
            sc_pc = None
            sc_pp = None
            for seg in pc_vals:
                if seg['start'] <= pos < seg['end']:
                    sc_pc = float(seg.get('score', None)) if 'score' in seg else None
                    break
            for seg in pp_vals:
                if seg['start'] <= pos < seg['end']:
                    sc_pp = float(seg.get('score', None)) if 'score' in seg else None
                    break
            out.append({"pos": int(pos), "phastCons": sc_pc, "phyloP": sc_pp})
        if len(out) > thinning:
            step = max(1, len(out)//thinning)
            out = out[::step]
        return out
    except Exception:
        return []


# 5) rCRS GenBank features → gold human windows (big boost)

In [ ]:
# ==== 5) rCRS GENBANK → GOLD HUMAN WINDOWS ====
# Produce many high-quality records for human_gene_type / human_gene_boundary

# Fetch full GenBank for rCRS (if not present)
gb_path = RAW/'references'/'rCRS.gb'
if not gb_path.exists():
    h = Entrez.efetch(db="nuccore", id="NC_012920.1", rettype="gb", retmode="text")
    gb_txt = h.read(); h.close()
    gb_path.write_text(gb_txt)

rec = SeqIO.read(str(gb_path), 'genbank')
seq = str(rec.seq).upper()

def slice_seq(s, a, b):
    a=max(0,int(a)); b=min(len(s),int(b))
    return s[a:b]

def mlaa_record_for_feature(start, end, strand, ftype, gname):
    start=int(start); end=int(end)
    ann = {
        "sequence": slice_seq(seq, start, end),
        "coords": {"ref":"rCRS","start":start,"end":end,"strand":"+" if strand in (1,None) else "-"},
        "gene_annotations": [{
            "type": ftype, "name": gname, "start": start, "end": end, "strand": ("+" if strand in (1,None) else "-")
        }],
        "structural_features": {"tRNA_structures": [], "rRNA_domains": [], "control_region_motifs": []},
        "functional_annotations": {"codon_usage": None, "regulatory_elements": [], "processing_sites": []},
        "variant_annotations": {"population_frequencies": [], "pathogenicity_scores": [], "clinical_significance": []},
        "evolutionary_context": {"conservation_scores": conservation_scores_chrM(start, end), "ortholog_mappings": [], "divergence_metrics": None},
        "sequence_features": {"gc": (slice_seq(seq,start,end).count('G')+slice_seq(seq,start,end).count('C'))/max(1,(end-start))},
        "expression_data": None
    }
    return ann

gold=[]
for feat in rec.features:
    if feat.type in ('CDS','tRNA','rRNA'):
        start = int(feat.location.start)
        end   = int(feat.location.end)
        strand= feat.strand
        gname = (feat.qualifiers.get('gene') or feat.qualifiers.get('product') or [''])[0]
        mlaa = mlaa_record_for_feature(start, end, strand, feat.type, gname)
        gold.append({"task":"human_gene_type","tier":"gold","source":"refseq","species":"Homo_sapiens","date":TODAY,"mlaa_json": to_json(mlaa)})
        gold.append({"task":"human_gene_boundary","tier":"gold","source":"refseq","species":"Homo_sapiens","date":TODAY,"mlaa_json": to_json(mlaa)})

# Add control-region slices as gold regulatory
for locus in CR_LOCI + REPL_ORIGINS:
    s, e = locus["start"], locus["end"]
    if s <= e:
        window = slice_seq(seq, s, e)
        ml = {
          "sequence": window,
          "coords": {"ref":"rCRS","start":s,"end":e,"strand":"+"},
          "gene_annotations": [],
          "structural_features": {"tRNA_structures": [], "rRNA_domains": [], "control_region_motifs":[locus["name"]]},
          "functional_annotations": {"codon_usage": None, "regulatory_elements": [locus["name"]], "processing_sites": []},
          "variant_annotations": {"population_frequencies": [], "pathogenicity_scores": [], "clinical_significance": []},
          "evolutionary_context": {"conservation_scores": conservation_scores_chrM(s,e), "ortholog_mappings": [], "divergence_metrics": None},
          "sequence_features": {"gc": (window.count('G')+window.count('C'))/max(1,(e-s))}
        }
        gold.append({"task":"human_regulatory","tier":"gold","source":"refseq","species":"Homo_sapiens","date":TODAY,"mlaa_json": to_json(ml)})

write_partitioned(gold, DATASET)
print("Added gold human windows:", len(gold))


# 6) Human MITOMAP → variant/regulatory (kept lightweight, scalable)

In [ ]:
# ==== 6) MITOMAP VARIANTS → HUMANS (gold/silver) ====
# We map disease-tagged positions as gold and others as silver for variant classification/regulatory hints.

def _row_to_mlaa_variant(row):
    s = int(row['start']); e = int(row['end'])
    window = slice_seq(rCRS_seq, s-1, e)  # GFF is 1-based inclusive; convert to 0-based half-open
    return {
        "sequence": window if window else "",
        "coords": {"ref":"rCRS","start":s-1,"end":e,"strand": row.get('strand','+')},
        "gene_annotations": [],
        "structural_features": {"tRNA_structures": [], "rRNA_domains": [], "control_region_motifs": []},
        "functional_annotations": {"codon_usage": None, "regulatory_elements": [], "processing_sites": []},
        "variant_annotations": {"population_frequencies": [], "pathogenicity_scores": [], "clinical_significance": [row.get('diseasename') or row.get('disease') or row.get('clinical_significance')]},
        "evolutionary_context": {"conservation_scores": conservation_scores_chrM(max(0,s-11), min(16569,e+10)), "ortholog_mappings": [], "divergence_metrics": None},
        "sequence_features": {"gc": (window.count('G')+window.count('C'))/max(1,len(window))}
    }

rows=[]
# disease GFF as gold
for _, r in dis.iterrows():
    rows.append({"task":"human_variant_cls","tier":"gold","source":"mitomap","species":"Homo_sapiens","date":TODAY,"mlaa_json": to_json(_row_to_mlaa_variant(r))})

# polymorphisms as silver (subsample to avoid explosion; take up to 20k)
poly_sample = poly.sample(min(20000, len(poly)), random_state=42) if len(poly)>0 else pd.DataFrame()
for _, r in poly_sample.iterrows():
    rows.append({"task":"human_variant_cls","tier":"silver","source":"mitomap","species":"Homo_sapiens","date":TODAY,"mlaa_json": to_json(_row_to_mlaa_variant(r))})

write_partitioned(rows, DATASET)
print("MITOMAP variant rows written:", len(rows))


# 7) Cross-species organelle GB → silver multispecies windows

In [ ]:
# ==== 7) CROSS-SPECIES FEATURE WINDOWS (silver) — robust seq resolver ====
from Bio.Seq import UnknownSeq

def iter_gb_files():
    for p in sorted((RAW/'ncbi_organelle').glob('*.gb')):
        yield p

def _safe_species_name(recx):
    sp = (recx.annotations.get('organism') or 'Unknown').strip()
    # compact and filesystem-safe
    sp = re.sub(r'\s+', '_', sp)
    sp = re.sub(r'[^A-Za-z0-9_.-]', '', sp)
    return sp or "Unknown"

def _resolve_record_sequence(recx, max_tries=2, sleep=0.5):
    """
    Return uppercase string sequence for a SeqRecord.
    If record carries UnknownSeq (no bases), fetch FASTA by accession via Entrez.
    Returns None if cannot resolve.
    """
    try:
        # Many records have a concrete sequence:
        if not isinstance(recx.seq, UnknownSeq):
            s = str(recx.seq)
            if s and all(c in "ACGTNacgtn" for c in s[:200]):  # cheap sanity
                return s.upper()
    except Exception:
        pass

    # Try to resolve by accession (use .id or .annotations['accessions'][0])
    acc = None
    cand = []
    if recx.id and len(recx.id) < 64:
        cand.append(recx.id)
    for a in recx.annotations.get('accessions', []):
        if a and a not in cand:
            cand.append(a)

    for acc in cand[:3]:  # try a few reasonable candidates
        for k in range(max_tries):
            try:
                h = Entrez.efetch(db="nuccore", id=acc, rettype="fasta", retmode="text")
                fa = h.read(); h.close()
                # Parse FASTA
                f = io.StringIO(fa)
                recs = list(SeqIO.parse(f, 'fasta'))
                if recs:
                    seq = str(recs[0].seq).upper()
                    if seq and len(seq) >= 100:  # org mtDNAs will be 2k–30k typically
                        return seq
            except Exception:
                pass
            time.sleep(sleep)

    return None  # give up if not resolvable

def _strand_char(loc_strand):
    return '+' if loc_strand in (1, None) else '-'

def _seq_gc_fraction(s):
    s = s.upper()
    return (s.count('G') + s.count('C')) / max(1, len(s))

ms = []
cnt = 0
flushed = 0
for gb in iter_gb_files():
    try:
        recs = list(SeqIO.parse(str(gb), 'genbank'))
    except Exception:
        continue

    for recx in recs:
        species = _safe_species_name(recx)
        sseq = _resolve_record_sequence(recx)
        if not sseq:
            # cannot work without bases
            continue

        # Keep whole-sequence sanity: organelle mtDNAs are usually 2k–30k
        if not (2000 <= len(sseq) <= 30000):
            # still allow partials, but be stricter in feature length below
            pass

        for feat in recx.features:
            if feat.type in ('CDS', 'tRNA', 'rRNA') and getattr(feat, 'location', None):
                try:
                    st = int(feat.location.start)
                    en = int(feat.location.end)
                    if en <= st:
                        continue
                    L = en - st
                    # keep reasonable gene sizes; allow a wider window, but bounded
                    if L < 20 or L > 6000:
                        continue
                    subseq = sseq[st:en]
                    if len(subseq) < 20:
                        continue
                    # skip only exact duplicates to preserve diversity
                    if dupfilter.exact_seen(subseq):
                        continue

                    gname = (feat.qualifiers.get('gene')
                             or feat.qualifiers.get('product')
                             or [''])[0]

                    ml = {
                        "sequence": subseq,
                        "coords": {
                            "ref": recx.id,
                            "start": st,
                            "end": en,
                            "strand": _strand_char(feat.location.strand)
                        },
                        "gene_annotations": [{
                            "type": feat.type,
                            "name": gname,
                            "start": st,
                            "end": en
                        }],
                        "structural_features": {
                            "tRNA_structures": [],
                            "rRNA_domains": [],
                            "control_region_motifs": []
                        },
                        "functional_annotations": {
                            "codon_usage": None,
                            "regulatory_elements": [],
                            "processing_sites": []
                        },
                        "variant_annotations": {
                            "population_frequencies": [],
                            "pathogenicity_scores": [],
                            "clinical_significance": []
                        },
                        "evolutionary_context": {
                            "conservation_scores": [],
                            "ortholog_mappings": [],
                            "divergence_metrics": None
                        },
                        "sequence_features": {"gc": _seq_gc_fraction(subseq)}
                    }

                    ms.append({
                        "task": "multispecies_gene_type",
                        "tier": "silver",
                        "source": "ncbi",
                        "species": species,
                        "date": TODAY,
                        "mlaa_json": to_json(ml)
                    })
                    cnt += 1
                    if cnt % 5000 == 0:
                        write_partitioned(ms, DATASET)
                        flushed += len(ms)
                        ms.clear()
                except Exception:
                    # skip malformed/compound locations we can't coerce
                    continue

# final flush
if ms:
    write_partitioned(ms, DATASET)
    flushed += len(ms)
    ms.clear()

print(f"Cross-species windows written: {cnt} (flushed: {flushed})")

# 8) Synthetic data (robust, scalable, chunked; relaxed dedup)

In [ ]:
# ==== 8) SYNTHETIC DATA (robust + chunked) ====
_ACGT = ('A','C','G','T'); _PUR={'A','G'}; _PYR={'C','T'}

def _safe_alt_for(base, rng):
    if base in _ACGT:
        return rng.choice([x for x in _ACGT if x != base])
    return rng.choice(_ACGT)

class SyntheticGenerator:
    def __init__(self, base_seq: str, rng_seed=1234):
        self.base = re.sub(r'[^ACGT]', 'N', str(base_seq).upper())
        self.rng = random.Random(rng_seed)
    def mutate(self, rate=1e-3, transitions_bias=3.0):
        seq = list(self.base); muts=[]
        for i, b in enumerate(seq):
            if self.rng.random() < rate:
                if b in _ACGT:
                    if b in _PUR:
                        cand = ['G'] if b=='A' else ['A']
                        others = [x for x in _ACGT if x not in (b, cand[0])]
                        pool = cand * int(transitions_bias) + others
                        alt = self.rng.choice(pool)
                    elif b in _PYR:
                        cand = ['T'] if b=='C' else ['C']
                        others = [x for x in _ACGT if x not in (b, cand[0])]
                        pool = cand * int(transitions_bias) + others
                        alt = self.rng.choice(pool)
                    else:
                        alt = _safe_alt_for(b, self.rng)
                else:
                    alt = _safe_alt_for(b, self.rng)
                if alt != b:
                    seq[i]=alt; muts.append((i,b,alt))
        return ''.join(seq), muts

def codon_usage(s):
    s = re.sub(r'[^ACGT]','',s)
    counts={}
    for i in range(0, len(s)-2, 3):
        c = s[i:i+3]
        counts[c]=counts.get(c,0)+1
    return counts

def cheap_feats(s):
    s=s.upper()
    gc = (s.count('G')+s.count('C'))/max(1,len(s))
    at_skew = (s.count('A')-s.count('T'))/max(1,(s.count('A')+s.count('T')))
    cg_skew = (s.count('C')-s.count('G'))/max(1,(s.count('C')+s.count('G')))
    return {"gc":gc,"at_skew":at_skew,"cg_skew":cg_skew}

n_target = CONFIG["synthetic_n"]
chunk=25_000
buf=[]
wrote=0
gen = SyntheticGenerator(rCRS_seq, rng_seed=42)

for i in range(n_target):
    s, muts = gen.mutate(rate=CONFIG["synthetic_mut_rate"], transitions_bias=3.0)
    if dupfilter.exact_seen(s):   # skip exact dup only (keep diversity)
        continue
    rec = {
      "sequence": s,
      "coords": {"ref":"synthetic","start":0,"end":len(s),"strand":"+"},
      "gene_annotations": [],
      "structural_features": {"tRNA_structures": [], "rRNA_domains": [], "control_region_motifs": []},
      "functional_annotations": {"codon_usage": {"table":"vertebrate_mito","counts": codon_usage(s)},
                                 "regulatory_elements": [], "processing_sites": []},
      "variant_annotations": {"population_frequencies": [], "pathogenicity_scores": [], "clinical_significance": []},
      "evolutionary_context": {"conservation_scores": [], "ortholog_mappings": [], "divergence_metrics": None},
      "sequence_features": cheap_feats(s),
      "expression_data": None,
      "meta": {"type":"synthetic_point_mut", "n_mut": len(muts)}
    }
    buf.append({"task":"synthetic_gene_type","tier":"silver","source":"synthetic","species":"synthetic","date":TODAY,"mlaa_json": to_json(rec)})
    if len(buf)>=chunk:
        write_partitioned(buf, DATASET); wrote+=len(buf); buf.clear()

if buf:
    write_partitioned(buf, DATASET); wrote+=len(buf); buf.clear()

print("Synthetic written:", wrote)


# 9) Curriculum manifest (patch Stage-1 so it’s never empty)

In [ ]:
# ==== 9) MANIFESTS (create/patch curriculum) ====
manifest = {
  "stages": [
    {
      "name": "stage1_gold",
      "globs": [
        "task=human_gene_type/tier=gold/**/*.parquet",
        "task=human_gene_boundary/tier=gold/**/*.parquet",
        # ensure stage1 has data even early on:
        "task=synthetic_gene_type/tier=silver/**/*.parquet"
      ]
    },
    {
      "name": "stage2_mix",
      "globs": [
        "task=human_gene_type/tier=gold/**/*.parquet",
        "task=human_gene_type/tier=silver/**/*.parquet",
        "task=human_variant_cls/tier=gold/**/*.parquet",
        "task=multispecies_gene_type/tier=silver/**/*.parquet"
      ]
    },
    {
      "name": "stage3_bronze_synth",
      "globs": [
        "task=human_gene_type/tier=bronze/**/*.parquet",
        "task=multispecies_gene_type/tier=bronze/**/*.parquet",
        "task=synthetic_gene_type/tier=silver/**/*.parquet"
      ]
    }
  ]
}
MANIFEST.mkdir(parents=True, exist_ok=True)
(MANIFEST/'splits.curriculum.json').write_text(json.dumps(manifest, indent=2))
(MANIFEST/'dataset.manifest.json').write_text(json.dumps({
    "root": str(DATASET),
    "tokenizer_dir": str(TOKDIR),
    "created": TODAY
}, indent=2))
print("Manifest written.")


# 10) Integrity + stats (metadata-only; fast & robust)

In [ ]:
# ==== 10) INTEGRITY + STATS (metadata-only) ====
def _parse_hive_parts(path, wanted=("task","tier","source","species","date")):
    parts={}
    for seg in Path(path).parts:
        if '=' in seg:
            k,v = seg.split('=',1)
            if k in wanted: parts[k]=v
    return parts

def scan_integrity(dataset_root: Path, out_csv: Path):
    rows=[]
    for p in dataset_root.rglob('*.parquet'):
        st = p.stat()
        rows.append({"file": str(p.relative_to(dataset_root)), "bytes": st.st_size, "mtime": int(st.st_mtime)})
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv, 'w', newline='') as f:
        w=csv.DictWriter(f, fieldnames=["file","bytes","mtime"])
        w.writeheader(); w.writerows(rows)

def dataset_stats(dataset_root: Path):
    by_task   = {}
    by_tier   = {}
    by_source = {}
    species_set = set()
    total_rows = 0
    files = list(dataset_root.rglob('*.parquet'))
    for f in files:
        try:
            meta = pq.ParquetFile(f).metadata
            n    = meta.num_rows or 0
        except Exception:
            n = 0
        total_rows += n
        parts = _parse_hive_parts(f)
        tsk   = parts.get("task","unknown")
        tier  = parts.get("tier","unknown")
        src   = parts.get("source","unknown")
        sp    = parts.get("species")
        by_task[tsk]   = by_task.get(tsk,0)+n
        by_tier[tier]  = by_tier.get(tier,0)+n
        by_source[src] = by_source.get(src,0)+n
        if sp: species_set.add(sp)
    stats = {
        "n_rows": int(total_rows),
        "by_task": by_task,
        "by_tier": by_tier,
        "by_source": by_source,
        "n_species": len(species_set),
        "n_files": len(files)
    }
    REPORTS.mkdir(parents=True, exist_ok=True)
    (REPORTS/'stats.json').write_text(json.dumps(stats, indent=2))
    return stats

scan_integrity(DATASET, REPORTS/'integrity.csv')
stats = dataset_stats(DATASET)
print(json.dumps(stats, indent=2))

{
  "n_rows": 66629,
  "by_task": {
    "human_gene_type": 111,
    "human_gene_boundary": 111,
    "human_regulatory": 36,
    "human_variant_cls": 61371,
    "multispecies_gene_type": 5000
  },
  "by_tier": {
    "gold": 3465,
    "silver": 63164
  },
  "by_source": {
    "refseq": 258,
    "mitomap": 61371,
    "ncbi": 5000
  },
  "n_species": 80,
  "n_files": 95
}
